# GLM-OCR 표 추출 시험 (Colab T4, Ollama)

GLM-OCR 하나만 10쪽에 돌린다. 지난 실행에서 표가 0건이라, 먼저 한 장을 직접 호출해 **빈 응답의 원인**을 가른 뒤 벤치를 돌린다.

- 런타임: T4 GPU
- Drive `MyDrive/bench/.pdfs/vlm-trial-10p.pdf`가 이미 있으면 그대로 쓴다. 없으면 3번 셀이 업로드를 받는다


In [ ]:
# 1. Drive · 벤치 데이터 위치
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys
from google.colab import drive
drive.mount('/content/drive')
BENCH_DATA = '/content/drive/MyDrive/bench'
os.makedirs(f'{BENCH_DATA}/.pdfs', exist_ok=True)
os.environ['BENCH_DATA'] = BENCH_DATA


In [ ]:
# 2. 코드
%cd /content
!rm -rf Policy-Chunker && git clone -q https://github.com/SM17-YoonDongJu/Policy-Chunker
%cd /content/Policy-Chunker
!pip install -q -e .


In [ ]:
# 3. 10쪽 PDF — Drive에 있으면 재사용, 없으면 전체 PDF를 받아 자른다
import pymupdf
TRIAL = f'{BENCH_DATA}/.pdfs/vlm-trial-10p.pdf'
PNG_DIR = f'{BENCH_DATA}/trial-pages'

if not os.path.exists(TRIAL):
    from google.colab import files
    import shutil
    up = files.upload()
    SRC = f'{BENCH_DATA}/.pdfs/source.pdf'
    shutil.move(next(iter(up)), SRC)

    from insurance_chunker import extractor
    doc = pymupdf.open(SRC)
    nums = list(range(1, doc.page_count + 1))
    sources, gap, _ = extractor.extract_cpu_tables(SRC, nums, use_vision=True)
    pym = sorted(sources['pymupdf'])
    plain = [p for p in nums if p not in pym and p not in gap]
    longest = sorted(pym, key=lambda p: -sources['pymupdf'][p].count('\n'))
    PAGES = pym[:4] + gap[:3] + [p for p in longest if p not in pym[:4]][:2] + plain[:1]
    print("PAGES =", PAGES, " (바꾸려면 여기서 고치고 다시 실행)")
    doc.select([p - 1 for p in PAGES]); doc.save(TRIAL); doc.close()

os.makedirs(PNG_DIR, exist_ok=True)
trial = pymupdf.open(TRIAL)
N = trial.page_count
for i, page in enumerate(trial, 1):
    page.get_pixmap(dpi=110).save(f'{PNG_DIR}/p{i:02d}.png')
print(TRIAL, N, "쪽")


In [ ]:
# 4. Ollama 설치 · 기동 · glm-ocr · GPU 확인
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import shutil, subprocess, time, requests
OLLAMA = shutil.which('ollama') or '/usr/local/bin/ollama'
assert os.path.exists(OLLAMA), "ollama 바이너리가 없다 — 설치 로그 확인"
subprocess.Popen([OLLAMA, 'serve'], stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://localhost:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open('/tmp/ollama.log').read()[-2000:])

!ollama pull glm-ocr:latest
!grep -i "inference compute" /tmp/ollama.log    # library=cuda name="Tesla T4" 여야 한다


In [ ]:
# 5. 진단 — 한 장을 두 경로로 직접 호출해 원시 응답을 본다
import base64, json
def b64(i, dpi=150):
    pix = pymupdf.open(TRIAL)[i - 1].get_pixmap(dpi=dpi)
    return base64.b64encode(pix.tobytes('png')).decode()

def via_openai(img, prompt='Table Recognition:', model='glm-ocr:latest'):
    r = requests.post('http://localhost:11434/v1/chat/completions', json={
        'model': model, 'temperature': 0, 'max_tokens': 2048,
        'messages': [{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{img}'}},
            {'type': 'text', 'text': prompt}]}]}, timeout=300)
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content']

def via_native(img, prompt='Table Recognition:', model='glm-ocr:latest'):
    r = requests.post('http://localhost:11434/api/generate', json={
        'model': model, 'prompt': prompt, 'images': [img], 'stream': False,
        'options': {'temperature': 0, 'num_predict': 2048}}, timeout=300)
    r.raise_for_status()
    return r.json().get('response', '')

img = b64(1)
a = via_openai(img); b = via_native(img)
print(f"A OpenAI 호환: {len(a)}자\n{a[:800]}\n")
print(f"B 네이티브:    {len(b)}자\n{b[:800]}")
!ollama ps


**읽는 법**
- A가 비고 B에 `<table>`이나 `|` 표가 있다 → Ollama OpenAI 호환 경로가 이 모델의 이미지를 못 넘긴다. 6번 프록시 셀을 켜고 7번에서 `VLM_URL`을 11435로 돌린다
- A·B 모두 표가 아닌 본문 텍스트거나 빈 문자열 → 페이지 전체 입력이 문제. 표 영역을 잘라 넣는 레이아웃 단계가 필요하다(코드 작업)
- A·B 모두 표가 나온다 → 벤치 쪽 문제. 7번 셀 결과의 `outcomes`를 본다


In [ ]:
# 5b. 전 페이지 원시 응답 훑기 (어느 경로가 되는지 확인된 뒤) — 페이지당 몇 초
USE = via_native            # A가 됐으면 via_openai 로
for i in range(1, N + 1):
    out = USE(b64(i))
    kind = 'HTML표' if '<table' in out.lower() else ('파이프표' if '|' in out else ('빈 응답' if not out.strip() else '표 아님'))
    print(f"p{i:02d} {len(out):5d}자 {kind:5s} | {out[:120].replace(chr(10), ' ')}")


In [ ]:
# 6. (A가 빈 경우만) OpenAI 호환 → Ollama 네이티브 변환 프록시. 코드 수정 없이 벤치가 /api/generate 를 쓰게 한다
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class _Proxy(BaseHTTPRequestHandler):
    def log_message(self, *a): pass
    def do_POST(self):
        try:
            req = json.loads(self.rfile.read(int(self.headers.get('Content-Length', 0))))
            prompt, images = '', []
            for part in req['messages'][-1]['content']:
                if part.get('type') == 'text':
                    prompt += part['text']
                elif part.get('type') == 'image_url':
                    u = part['image_url']['url'] if isinstance(part['image_url'], dict) else part['image_url']
                    images.append(u.split(',', 1)[1])
            r = requests.post('http://localhost:11434/api/generate', json={
                'model': req.get('model'), 'prompt': prompt, 'images': images, 'stream': False,
                'options': {'temperature': req.get('temperature', 0),
                            'num_predict': req.get('max_tokens', 4096)}}, timeout=600).json()
            body = {'choices': [{'message': {'role': 'assistant', 'content': r.get('response', '')},
                                 'finish_reason': 'length' if r.get('done_reason') == 'length' else 'stop'}],
                    'usage': {'prompt_tokens': r.get('prompt_eval_count'),
                              'completion_tokens': r.get('eval_count')}}
            out, code = json.dumps(body).encode(), 200
        except Exception as e:
            out, code = json.dumps({'error': str(e)}).encode(), 500
        self.send_response(code); self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(out))); self.end_headers(); self.wfile.write(out)

_srv = ThreadingHTTPServer(('127.0.0.1', 11435), _Proxy)
threading.Thread(target=_srv.serve_forever, daemon=True).start()
print("프록시 11435 →", via_openai.__name__, "대신 네이티브로 변환. 확인:")
r = requests.post('http://localhost:11435/v1/chat/completions', json={'model': 'glm-ocr:latest',
    'messages': [{'role': 'user', 'content': [
        {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{b64(1)}'}},
        {'type': 'text', 'text': 'Table Recognition:'}]}]}, timeout=300)
print(r.status_code, r.json()['choices'][0]['message']['content'][:300])


In [ ]:
# 7. 벤치 실행 — 문서셋 · 변형 작성 후 run
import yaml
from pathlib import Path
VLM_URL = 'http://localhost:11434'      # 6번 프록시를 켰으면 'http://localhost:11435'

Path('eval/bench/docsets/vlm-trial.yaml').write_text(yaml.safe_dump({
    'name': 'vlm-trial',
    'documents': [{'id': 'vlm-trial-10p', 'pdf': TRIAL, 'type': 'table',
                   'doc_type': 'policy_terms', 'insurer': '미상'}]}, allow_unicode=True), encoding='utf-8')
Path('eval/bench/variants/vlm-trial-glm.yaml').write_text(yaml.safe_dump({
    'slug': 'vlm-trial-glm', 'ref': 'WORKTREE', 'status': 'draft',
    'env': {'VLM_BACKEND': 'local', 'VLM_TARGET': 'both', 'VISION_MAX_PAGES': str(N),
            'VLM_CONCURRENCY': '1', 'VLM_TIMEOUT': '600',
            'VLM_URL': VLM_URL, 'VLM_MODEL': 'glm-ocr:latest',
            'VLM_PROMPT': 'Table Recognition:'}}, allow_unicode=True), encoding='utf-8')

!python eval/bench/bench.py run --docset vlm-trial --variant vlm-trial-glm --from vlm --to combine

rep = json.load(open(f'{BENCH_DATA}/results/vlm-trial-glm/vlm-trial/vlm-trial-10p.json'))
v = rep['stages']['vlm']
print({k: v.get(k) for k in ('calls', 'tables', 'outcomes', 'truncated', 'latency_p50', 'latency_p95')})
print('채택 소스:', rep['stages']['combine'].get('chosen'))


In [ ]:
# 8. 페이지별 결과 — 이미지 · PyMuPDF 표 · GLM-OCR 표
import glob
from IPython.display import display, Image, Markdown
from insurance_chunker.models import compute_doc_hash

doc_dir = Path(BENCH_DATA) / '.artifacts' / compute_doc_hash(TRIAL)[:16]
latest = lambda pat: max(glob.glob(str(doc_dir / pat)), key=os.path.getmtime)
pym = json.load(open(latest('tables_cpu-*.json')))['output']['sources'].get('pymupdf', {})
glm = {}
for f in glob.glob(str(doc_dir / 'vlm-*.json')):
    d = json.load(open(f))
    if d.get('config', {}).get('model') == 'glm-ocr:latest':
        glm = d['output']; break

lines = ['# GLM-OCR 10쪽 결과', '']
for i in range(1, N + 1):
    p = str(i)
    lines += [f'## p{i}', '', f'![p{i}](trial-pages/p{i:02d}.png)', '',
              '### PyMuPDF', '', pym.get(p) or '(없음)', '',
              '### GLM-OCR', '', glm.get(p) or '(출력 없음)', '']
Path(f'{BENCH_DATA}/glm-ocr-trial.md').write_text('\n'.join(lines), encoding='utf-8')
print("저장:", f'{BENCH_DATA}/glm-ocr-trial.md')

for i in range(1, N + 1):
    display(Image(f'{PNG_DIR}/p{i:02d}.png', width=480))
    display(Markdown(f"**p{i} GLM-OCR**\n\n" + (glm.get(str(i)) or '(출력 없음)')))
